# VALENCE v0.7.0 — Colab Quickstart

This notebook uses either a VALENCE source tree already stored in Google Drive
or a public GitHub repository URL. It does not embed a duplicate copy of the
repository.


In [ ]:
from google.colab import drive
from pathlib import Path
import shutil
import subprocess
import sys

drive.mount('/content/drive')

# After the GitHub repository is published, place its clone URL here.
REPO_URL = ''

DRIVE_ROOT = Path('/content/drive/MyDrive/valence')
DRIVE_SOURCE = DRIVE_ROOT / 'source'
RESULTS_ROOT = DRIVE_ROOT / 'results' / 'public-v0.7.0'
RUNTIME_CLONE = Path('/content/valence')

if (DRIVE_SOURCE / 'pyproject.toml').exists():
    PROJECT_ROOT = DRIVE_SOURCE
elif REPO_URL:
    if RUNTIME_CLONE.exists():
        shutil.rmtree(RUNTIME_CLONE)
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'v0.7.0', REPO_URL, str(RUNTIME_CLONE)], check=True)
    PROJECT_ROOT = RUNTIME_CLONE
else:
    raise RuntimeError(
        'No Drive source was found. Set REPO_URL after the public GitHub repository is created.'
    )

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
print(f'Project: {PROJECT_ROOT}')
print(f'Results: {RESULTS_ROOT}')


In [ ]:
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', '--no-build-isolation', '-e', f'{PROJECT_ROOT}[dev,paper]'],
    check=True,
)
subprocess.run([sys.executable, '-m', 'pytest', '-q', str(PROJECT_ROOT / 'tests')], cwd=PROJECT_ROOT, check=True)


In [ ]:
for config_name in ("smoke.yaml", "outage_demo.yaml", "finality_demo.yaml"):
    output = RESULTS_ROOT / config_name.removesuffix(".yaml").replace("_", "-")
    if output.exists():
        shutil.rmtree(output)
    subprocess.run(
        ["valence", "run", str(PROJECT_ROOT / "configs" / config_name), "--output", str(output)],
        cwd=PROJECT_ROOT,
        check=True,
    )
    print(f"Completed {config_name}: {output}")

subprocess.run(
    [sys.executable, str(PROJECT_ROOT / "scripts" / "run_poster_compliance_demo.py"),
     "--output", str(RESULTS_ROOT / "poster-compliance")],
    cwd=PROJECT_ROOT,
    check=True,
)


## Optional full reproduction

Set `RUN_FULL_EXPERIMENTS = True` to run the four frozen experiment pipelines
and generate paper figures. This is more computationally expensive than the
smoke test.


In [ ]:
RUN_FULL_EXPERIMENTS = False

if RUN_FULL_EXPERIMENTS:
    v06 = RESULTS_ROOT / 'v0.6'
    commands = [
        ('run_p99_dose_sweep.py', v06 / 'p99-dose'),
        ('run_temporal_dependence_experiment.py', v06 / 'temporal'),
        ('run_beyond_p99_experiment.py', v06 / 'beyond-p99'),
        ('run_finality_frontier.py', v06 / 'finality-frontier'),
    ]
    for script, output in commands:
        subprocess.run(
            [sys.executable, str(PROJECT_ROOT / 'scripts' / script), '--output', str(output)],
            cwd=PROJECT_ROOT,
            check=True,
        )
    subprocess.run(
        [
            sys.executable,
            str(PROJECT_ROOT / 'scripts' / 'build_paper_artifacts.py'),
            '--results-root', str(v06),
            '--output', str(RESULTS_ROOT / 'paper-artifacts-v0.6'),
        ],
        cwd=PROJECT_ROOT,
        check=True,
    )
    print('Full reproduction complete.')
else:
    print('Full experiments skipped. Set RUN_FULL_EXPERIMENTS = True to run them.')
